In [114]:
import numpy as np
import pandas as pd

1) [5 points] Divide the data into train / test sets (80% and 20% respectively)

In [115]:
# load data into a pandas dataframe for easy processing
data = pd.read_csv('hw4_trainingset.csv')
data.head()
data_copy = data.copy()
training = data_copy.sample(frac=0.8)
test_set = data_copy.drop(training.index)
training

,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Label
1210,12627.80,46045.90,-5891.05,71,273,2061,0
4219,7128.65,13917.60,-2441.00,93,120,809,0
4924,9715.95,30500.20,-3205.04,122,171,1747,1
1128,7981.00,19148.00,-3627.00,17,143,1061,1
5452,13877.90,141412.00,-6162.47,175,308,6057,0
...,...,...,...,...,...,...,...
95,3749.37,2140.32,-548.99,6,43,161,1
4392,5045.71,4259.90,-1600.00,13,88,292,1
5357,10878.00,25961.00,-4002.00,28,178,1208,1
2060,18307.70,196636.00,-8460.00,38,390,7459,0


2) (25points)Implement a Multinomial Naïve Bayes classifierfrom scratch, with smoothing (you can setthe smoothing value to 1).You are free to code this up however you like, however, make sure that there is a function that can be called with a test X vector and returns the predicted Y.

In [116]:
# Divide training set into validation set and train sets
training_copy = training.copy()
train_set = training_copy.sample(frac = 0.75)
valid_set = training_copy.drop(train_set.index)
# test_set_head = test_set.head()
# valid_set.head()
# Feature_1 = train_set["Feature_1"]
# Feature_2 = train_set["Feature_2"]
# Feature_3 = train_set["Feature_3"]
# Feature_4 = train_set["Feature_4"]
# Feature_5 = train_set["Feature_5"]
# Feature_6 = train_set["Feature_6"]
# Label = train_set["Label"]
# Feature_6.head()

# create a rows of vectors to represent the data for training
def create_row_vectors(dataframe):
    row_vectors = [] ## start with an empty list
    for i , line in dataframe.iterrows():
        # create a list of that row
        row_vector = [line.Feature_1,line.Feature_2,line.Feature_3,line.Feature_4,line.Feature_5,line.Feature_6,
                      line.Label]
        
        # put list in overall list to be returned
        row_vectors.append(row_vector)
        
    return row_vectors

# calculate the probability of each class
def cal_class_probs(training_list):
    class_probabilities = {}
    zero_class = 0
    one_class = 0
    len_train = len(training_list)
    
    for vector in training_list:
        if vector[-1] == 0:
            zero_class += 1
        elif vector[-1] == 1:
            one_class += 1
    
    class_probabilities[0.0] = zero_class/len_train
    class_probabilities[1.0] = one_class/len_train
    
    return class_probabilities,zero_class,one_class  # return number of classes

# sort feature for bucketing
def sort_by_feature(dataframe,feature):
    sorted_df = dataframe.sort_values(by=feature)
    return sorted_df

def partition_feature(bucket_size,sorted_feature,feature_num):
    bucket_and_probs = {}
    buc_range_dict = {}
    
    ## sample and calculate probabilty of class0 or 1 if in each sampled range
    samples = round(sorted_feature.shape[0]/bucket_size)
    start = 0
    end = bucket_size
    overall = create_row_vectors(sorted_feature) 

    for i in range(samples):
        # get bucket
        zero_class = 0
        one_class = 0
        feature_sample = sorted_feature[start:end]
        
        ## process dataframe and calculate rows that has labels 0 or 1
        ## first convert into list of rows
        row_vectors = create_row_vectors(feature_sample)
        
        # get range based on bucket sample size
        min_feature = overall[i*bucket_size][feature_num-1]
        if ((i*bucket_size)+bucket_size < len(overall)):
            max_feature = overall[(i*bucket_size)+bucket_size][feature_num-1]
        else:
             max_feature = overall[-1][feature_num-1]
        
        # create a dictionary of bucket number and corresponding range
        buc_range_dict[i] = [min_feature,max_feature]
        
        # go through rows and count 0 and 1s for each feature
        for vector in row_vectors:
            if vector[-1] == 0:
                zero_class += 1
            elif vector[-1] == 1:
                one_class += 1
                
        start = start + bucket_size
        end = end + bucket_size
        
        # keep track of number of classes in each bucket
        bucket_and_probs[i] = [zero_class,one_class]

        
    return bucket_and_probs,buc_range_dict   
    
def find_bucket(number,buc_range_dict):
    for key in buc_range_dict:
        if (number >= buc_range_dict[key][0]) and (number < buc_range_dict[key][1]):
            return key 
    
    # return last bucket if range not captured
    return (len(buc_range_dict)-1)

    
    
def cal_feature_prob(vector,class_probs,dataframe,bucket_size):
    class_0p = float(class_probs[0][0])
    class_1p = float(class_probs[0][1])
    total_0 = class_probs[1]
    total_1 = class_probs[2]
    probabilities = {}
    probabilities[0.0] = 1
    probabilities[1.0] = 1
    
    ## for the vector given predict a label for every feature
    for i in range(len(vector)-1):
        
        feature = sort_by_feature(dataframe,"Feature_" + str(i+1))
        buc_and_probs1, buc_range_dict1 = partition_feature(bucket_size,feature,i+1)
        buck = find_bucket(vector[i],buc_range_dict1)
        probabilities[0.0] +=  np.log((buc_and_probs1[buck][0]+1)/(total_0 + 2))
        probabilities[1.0] +=  np.log((buc_and_probs1[buck][1]+1)/(total_1 + 2))
        
    probabilities[0.0] += np.log(class_0p)
    probabilities[1.0] += np.log(class_1p)
    
    
    if probabilities[0.0] >= probabilities[1.0]:
        return 0.0
    else:
        return 1.0
    
        
def make_predictions(test_vectors,class_probs,train,bucket_size):
    # return a list of predictions made for a given test data
    predicted = []
    predicted_n = 0
    
    for feature in test_vectors:
        predicted_label = cal_feature_prob(feature,class_probs,train,bucket_size)
        predicted.append(predicted_label)
        predicted_n += 1
        print(predicted_label)
        
    
    return predicted  
         
def cal_accuracy(test_data,predicted):
    correct = 0
    
    for i in range(len(predicted)):
        if (predicted[i] == test_data[i][-1]):
            correct += 1
    return correct/len(predicted)       
     
    
    

In [117]:
## overall list
train_list = create_row_vectors(train_set)
class_probs_and_tots = cal_class_probs(train_list)
test_vectors = create_row_vectors(test_set[0:30])  # test on these amount of vectors because of implementation
# can change size of test by playing around with range

predictedMN = make_predictions(test_vectors,class_probs_and_tots,train_set,100)

print("Accuracy of Multinomial :=",cal_accuracy(test_vectors,predictedMN))



1.0
1.0
0.0
1.0
0.0
1.0
1.0
0.0
1.0
0.0
1.0
0.0
0.0
0.0
1.0
0.0
0.0
0.0
0.0
0.0
1.0
0.0
1.0
0.0
0.0
1.0
0.0
0.0
0.0
1.0
Accuracy of Multinomial := 0.6333333333333333


3)(25 points)Implement a Gaussian Bayes classier from scratch. 

In [118]:
def create_row_vectors(dataframe):
    row_vectors = [] ## start with an empty list
    for i , line in dataframe.iterrows():
        # create a list of that row
        #print(dataframe.Feature_1)
        row_vector = [line.Feature_1,line.Feature_2,line.Feature_3,line.Feature_4,line.Feature_5,line.Feature_6,
                      line.Label]
        # put list in overall list to be returned
        row_vectors.append(row_vector)
    return row_vectors

# create a dictionary by class
def separate_class(dataset):
    # create dictionary
    classes = {}
    #iterate through dataset
    for vector in dataset:
        # create a dictionary with labels as keys
        label = vector[-1]
        if (label not in classes):
            classes[label] = []
            classes[label].append(vector)
        else:
            classes[label].append(vector)
    return classes  

# for Gaussian distribution, calculate the means and variance
def get_mean(data):
    # convert data into numpy array
    data_arr = np.array(data)
    # return mean for all features
    mean = np.mean(data_arr,axis = 0)
    return mean[:-1]

def get_variance(data):
    # convert data into numpy array
    data_arr = np.array(data)
    # return variance of each feature
    variance = np.var(data_arr,axis = 0)
    return variance[:-1]

# compute mean and variance for each feature in a given class
def summary_class(data):
    # separate by class
    classes = separate_class(data)
    # a dictionary that holds the means and variance of every feature in a class
    mean_var_by_class = {}
    for label, vector in classes.items():
        mean_var_by_class[label] = get_mean(vector),get_variance(vector)
    return  mean_var_by_class

# function to calculate probability of each feature,given class mean and deviation
def cal_gaussian_prob(feature,mean_feature,variance_feature):
    # now, calculate the class probabilities of a given input vector
    # find the exponent part
    power = (-1*(feature - mean_feature)**2)/(2*variance_feature)
    norm = np.sqrt(2*np.pi*variance_feature)
    g_prob = (1/norm)*np.exp(power)
    return g_prob

# next we create a function that computes the overall probability of a given feature vector
def cal_feature_class_probs(feature,mean_var_by_class):
    feature = feature[:-1]
    probabilities = {}
    for label , mean_var in mean_var_by_class.items():
        probabilities[label] = 1 # start with 100% probability and multiply them out
        # iterate through the mean and variance of each feature and use it to calc prob
        
        mean = mean_var[0]
        variance = mean_var[1]

        for i in range(len(mean)):
            probabilities[label] *= cal_gaussian_prob(feature[i],mean[i],variance[i])
            
    return probabilities

def select_best_label(feature,mean_var_by_class):
    probabilities = cal_feature_class_probs(feature,mean_var_by_class)
    bestLabel = -1
    best_prob = -1
    #print("pbx",probabilities.items())
    for label, prob in probabilities.items():
        #print("pbx",probabilities.items())
        if (bestLabel < -1) or (prob > best_prob):
            bestLabel = label
            best_prob = prob
    return bestLabel
    
def make_predictions(test_data, mean_var_by_class):
    # return a list of predictions made for a given test data
    predicted = []
    for feature in test_data:
        predicted.append(select_best_label(feature,mean_var_by_class))
        #print("predicted:",predicted[0])
    return predicted

def cal_accuracy(test_data,predicted):
    correct = 0
    for i in range(len(predicted)):
        if (predicted[i] == test_data[i][-1]):
            correct += 1
    return correct/len(predicted)

In [120]:
train_list = create_row_vectors(training)
classes = separate_class(train_list)
mvc = summary_class(train_list)
test_list = create_row_vectors(test_set)
predicted = make_predictions(test_list, mvc)
print("Accuracy GaussianNB:=",cal_accuracy(test_list, predicted))

Accuracy GaussianNB:= 0.6151785714285715


4)(10points)Calculate the accuracy and the F1 score of test data using both of your models implemented above.

In [121]:
def F1_score(test_data,predicted):
    ## Your code here
    ## compare actual and predicted and build : y_actual and Y_pred are lists
    false_positive = 0
    false_negative = 0
    true_positive = 0
    true_negative = 0
    for i in range(len(predicted)):
        
        if predicted[i] == 0:
            if test_data[i][-1] == 0:
                true_negative += 1
            elif test_data[i][-1] == 1:
                false_negative += 1
                
        elif predicted[i] == 1:
            if test_data[i][-1] == 0:
                false_positive += 1
            elif test_data[i][-1] == 1:
                true_positive += 1
   
    recall = true_positive/(true_positive + false_negative)
    precision = true_positive/(true_positive + false_positive)
    F1 = 2*(precision*recall)/(precision + recall)
    return F1

In [122]:
print("F1 GaussianNB :=",F1_score(test_list,predicted))
print("F1 MultinomialNB :=",F1_score(test_vectors,predictedMN))

F1 GaussianNB := 0.3557548579970105
F1 MultinomialNB := 0.5599999999999999


### Bonus points(20points)
4) Implement the  two  models above using the  scikit-learn  library(https://scikit-learn.org/stable/modules/naive_bayes.html#)instead of coding it from scratch. Compare the results of the scikit-learn library(F1 and accuracy)to yourown implementationon the test set. (Printing the performance number is fine, you do not need tocreate any figures).Note that there might be differences between your models’performance and the performance of the scikit-learn library, do not worry about that,you will still get full grade. 

### SK Gaussian

In [123]:
import sklearn
from sklearn.model_selection import train_test_split 
from sklearn.naive_bayes import *

data = pd.read_csv('hw4_trainingset.csv')
data['Feature_3'] = data['Feature_3'].abs()
data.head()
data_copy = data.copy()
training = data_copy.sample(frac=0.8)

test_set = data_copy.drop(training.index)
train_y = training["Label"]
actual_label = test_set["Label"]
trainingX = training.drop("Label",axis =1)
testX = test_set.drop("Label",axis =1)


In [124]:
X= data.drop('Label',axis=1)
y = data['Label']
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2)

G_model = GaussianNB()
G_model.fit(X_train, y_train)
y_pred = G_model.predict(X_test)

In [125]:
# Metrics
G_accuracy = sklearn.metrics.accuracy_score(y_pred,y_test)
print("Accuracy GuassianNB[sklearn] :=",G_accuracy)
f1_score = sklearn.metrics.f1_score(y_pred,y_test)
print("F1 score of GuassianNB[sklearn] :=",f1_score)

Accuracy GuassianNB[sklearn] := 0.5892857142857143
F1 score of GuassianNB[sklearn] := 0.27672955974842767


In [79]:
MN_model = MultinomialNB()
MN_model.fit(X_train, y_train)
y_pred = MN_model.predict(X_test)

In [127]:
MN_accuracy = sklearn.metrics.accuracy_score(y_pred,y_test)
print("Accuracy MultinomialNB [sklearn] :=",MN_accuracy)
f1_score = sklearn.metrics.f1_score(y_pred,y_test)
print("F1 score MultinomialNB [sklearn] :=",f1_score)

Accuracy MultinomialNB [sklearn] := 0.5892857142857143
F1 score MultinomialNB [sklearn] := 0.27672955974842767


### Summary
#### Accuracy and F1 for MN
Multinomial Naive Bayes: Accuracy of my model was predicted to be $0.63333$ and that for the sklearn was $0.5892857142857143.$
The F1 score for my model was $0.5599999999999999$ and that of sklearn was that for the sklearn was $0.27672955974842767$

#### Accuracy and F1 for Gaussian
Gaussian Naive Bayes: Accuracy of my model was predicted to be $0.6151785714285715$ and that for the sklearn was 0.5892857142857143
The F1 score for my model was 0.3557548579970105 and that of sklearn was that for the sklearn was $0.27672955974842767$
